# 技能3 · Day 1 上机：用实验基准审计观测因果估计

**版本**：v5.0 + CQ-S3-1 质量补强
**配套**：notes.md（讲义）｜ data/README.md（数据来源）｜ starter.ipynb（TODO版）

## 学习目标
学完你能：
1. 区分 `nsw_mixtape` 随机实验样本与 `cps_mixtape` 观测对照，并用实验均值差建立基准
2. 用 DoWhy 完成“建模→识别→估计→反驳”，同时检查 overlap 与识别假设
3. 比较观测朴素值、调整估计和实验基准，并正确解释 refuter 不能证明因果为真

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

In [ ]:
# !pip install causaldata dowhy econml -q

## 1. 数据集背景与营销映射

本上机明确区分两个样本：`nsw_mixtape` 含 NSW 随机实验的处理组与实验对照组，可给出实验基准；`cps_mixtape` 是观测对照库。我们把 NSW 处理组与 CPS 对照组拼成观测样本，专门演示选择偏差，不能把实验对照与观测对照混为一谈。

| 数据变量 | 营销映射 | 角色 |
|---------|---------|------|
| `treat` | 是否收到优惠券/看到广告 | 处理 T |
| `re78` | 转化率 / GMV / 客单价 | 结果 Y |
| `age`,`educ`,`re74`,`re75`,`black`,`hisp`,`marr`,`nodegree` | 用户画像 / 历史消费 | 处理前协变量 X |

**因果问题**：观测样本的朴素均值差与 NSW 实验基准相差多少？在 overlap 允许的范围内，后门调整能否缩小这一差距？

In [ ]:
import pandas as pd
import numpy as np
import dowhy
from dowhy import CausalModel
from causaldata import nsw_mixtape, cps_mixtape

## 1-2：加载与探索真实数据

In [ ]:
# 1. 分别加载实验样本与观测对照，再构造观测比较样本
nsw_df = nsw_mixtape.load_pandas().data.copy()
cps_df = cps_mixtape.load_pandas().data.copy()
rct_df = nsw_df.copy()
df = pd.concat([
    nsw_df.loc[nsw_df["treat"] == 1],
    cps_df.loc[cps_df["treat"] == 0],
], ignore_index=True)

print(f"NSW 实验样本: {rct_df.shape}")
print(f"NSW处理组 + CPS观测对照: {df.shape}")
df.head()

In [ ]:
# 2. 探索观测样本的协变量平衡；SMD 比原始均值差更可比
print("处理组样本量:", len(df[df["treat"] == 1]))
print("观测对照组样本量:", len(df[df["treat"] == 0]))

covariates = ["age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"]

def smd(frame, column):
    treated = frame.loc[frame["treat"] == 1, column]
    control = frame.loc[frame["treat"] == 0, column]
    pooled_sd = np.sqrt((treated.var(ddof=1) + control.var(ddof=1)) / 2)
    return (treated.mean() - control.mean()) / pooled_sd if pooled_sd else 0.0

balance = df.groupby("treat")[covariates].mean().T
balance.columns = ["观测对照(treat=0)", "NSW处理(treat=1)"]
balance["SMD"] = [smd(df, column) for column in covariates]
print(balance)
print("|SMD| > 0.1 表示需要关注；还必须另画倾向得分 overlap，不能只凭均值表宣称可识别。")

## 2. 因果图（DAG）与样本选择

在 NSW 随机实验内部，处理由随机分配，协变量差异主要是有限样本波动；实验均值差可作为基准。把 NSW 处理组与 CPS 观测对照拼接后，`age/educ/re74/re75/...` 同时预测样本进入哪一组与结果，形成选择偏差。

```
X ──> sample/treat ──> re78
│                    ▲
└────────────────────┘
```

后门调整只有在 consistency、exchangeability、positivity 与 SUTVA 足够合理时才可识别目标效应。营销映射同理：历史活跃度既影响投放概率，也影响转化，但未观测动机仍可能破坏 exchangeability。

## 3：朴素估计（有偏）

In [ ]:
# 3. 先算随机实验基准，再算观测样本的朴素均值差
rct_ate = rct_df.loc[rct_df["treat"] == 1, "re78"].mean() - rct_df.loc[rct_df["treat"] == 0, "re78"].mean()
naive_ate = df.loc[df["treat"] == 1, "re78"].mean() - df.loc[df["treat"] == 0, "re78"].mean()
print(f"NSW 实验基准均值差 = {rct_ate:.2f}")
print(f"观测样本朴素均值差 = {naive_ate:.2f}")
print("两者差距是选择偏差的教学信号；它不是某个后门模型必然能完全消除的已知真值。")

## 3. 为什么观测比较会有偏

本例的偏差来自比较设计，而不是“真实数据天然有混杂”：NSW 实验组内的随机分配提供基准；NSW 处理组与 CPS 对照组来自不同选择机制，朴素比较混入了样本选择差异。后门调整尝试在已观测 X 上恢复可比性，但只有在共同支撑和无未观测混杂成立时才可能逼近实验基准。

**营销类比**：若优惠券由运营定向发给高活跃用户，收到与未收到的用户不是随机可比；控制历史活跃度能减少偏差，却不能自动消除未记录的投放规则。

## 4-5：DoWhy 因果分析

In [ ]:
# 4. DoWhy 建模 → 识别 → 估计（后门调整）
common_causes = ["age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"]

model = CausalModel(
    data=df,
    treatment="treat",
    outcome="re78",
    common_causes=common_causes,
)
identified_estimand = model.identify_effect()
causal_estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
)
print(f"后门调整估计 = {causal_estimate.value:.2f}")
print(f"观测朴素值 {naive_ate:.2f} | 实验基准 {rct_ate:.2f}")
print("调整值更接近实验基准是诊断信号，不等同于识别假设已被证明。")

In [ ]:
# 5. 反驳检验（安慰剂处理）
refutation = model.refute_estimate(
    identified_estimand,
    causal_estimate,
    "placebo_treatment_refuter"
)
print(refutation)
print()
print("解读：安慰剂处理下，新估计应接近 0 —— 若如此，说明方法没在虚假处理上'发现'效应，估计可靠。")

## 4. 营销延伸：倾向得分匹配（PSM）

后门调整（线性回归）是一种方法。PSM 是另一种常用的观测数据因果估计法：按"收到处理的概率"（倾向得分）把处理组与对照组匹配，再算匹配后的均值差。

在营销中，PSM 常用于把"收到优惠券的用户"与"相似但没收到优惠券的用户"匹配，估计优惠券的真实增量效应。

In [ ]:
# 6（可选）：PSM 再估一次
psm_estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_matching"
)
print(f"PSM 估计 ATE = {psm_estimate.value:.2f}")
print(f"三种估计对比：朴素 {naive_ate:.2f} | 后门回归 {causal_estimate.value:.2f} | PSM {psm_estimate.value:.2f}")
print()
print("解读：朴素估计与后门/PSM 估计的差异 = 混杂偏差。后门与 PSM 两种方法的接近程度 = 估计稳健性。")

## 5. 反思与前沿

### 反思问题
1. 朴素估计与后门调整估计的差异，主要来自哪个混杂变量？（提示：看 TODO2 的均衡性对比，哪 个协变量两组差距最大）
2. 安慰剂检验结果是否支持你的因果估计？（若安慰剂效应≈0，说明方法没在虚假处理上"发现"效应，方法可靠）
3. 如果 NSW 数据里有个**没观测到的混杂**（如"个人上进心"），你的估计还可靠吗？→ 这是"可忽略性"假设的根本局限

### 2026 前沿：LLM-as-a-judge 自检因果论证
把你建好的 DAG + 识别策略 + 估计 + 反驳结果整理成一段结构化描述，让 LLM 扮演"因果推断评审"，检查：
- DAG 是否遗漏了可能的混杂？
- 识别策略是否满足后门准则？
- 反驳检验是否充分？
- 结论是否过度外推？

参考 arXiv 2306.05685（NeurIPS 2023, LLM-as-a-judge）。**注意**：LLM 只审查论证质量，不估计效应本身——它停留在因果阶梯 L1，不能上升到 L2/L3。